# Maze Solver Training - Modular Version

This notebook uses modular code from `src/` folder:
- `src/model.py`: ResNet + GPT2 model architecture
- `src/tokenizer.py`: Simple tokenizer for maze sequences
- `src/dataset.py`: Dataset and DataLoader utilities
- `src/train_utils.py`: Training and evaluation functions

In [63]:
import sys
import json
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

# Add src to path
sys.path.append('./src')

# Import our modules
from model import ResNetGPT2PrefixModel
from tokenizer import SimpleTokenizer
from dataset import MazeDataset, collate_fn
from train_utils import train_model, test_model

print("All modules imported successfully")

All modules imported successfully


## 1. Setup Tokenizer

In [64]:
print("=" * 60)
print("TOKENIZER SETUP")
print("=" * 60)

tokenizer = SimpleTokenizer()

print(f"Vocabulary: {tokenizer.vocab}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Special tokens: BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}")

TOKENIZER SETUP
Vocabulary: {'<pad>': 0, '<s>': 1, '</s>': 2, '<unk>': 3, 'R': 4, 'U': 5}
Vocab size: 6
Special tokens: BOS=1, EOS=2, PAD=0


## 2. Load Data

In [65]:
print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

# Load JSON data
with open('data/train_sequences.json', 'r') as f:
    train_data = json.load(f)
    train_entries = train_data['entries']  # <-- Access the 'entries' key
    train_metadata = train_data['metadata']

with open('data/test_sequences.json', 'r') as f:
    test_data = json.load(f)
    test_entries = test_data['entries']  # <-- Access the 'entries' key
    test_metadata = test_data['metadata']

print(f"Training set: {len(train_entries)} examples")
print(f"Test set: {len(test_entries)} examples")

# Print metadata
print("\n" + "=" * 60)
print("DATASET METADATA")
print("=" * 60)
print(f"Grid size: {train_metadata['grid_size']}×{train_metadata['grid_size']}")
print(f"Start position: {train_metadata['start_position']}")
print(f"Goal position: {train_metadata['goal_position']}")
print(f"Variations per solution: {train_metadata['variations']}")
print(f"Seed: {train_metadata['seed']}")

# Verify sample
print("\n" + "=" * 60)
print("VERIFICATION - Sample Entry")
print("=" * 60)
sample = train_entries[0]
print(f"Sample ID: {sample['id']}")
print(f"Solution ID: {sample['solution_id']}")
print(f"Variation: {sample['variation']}")
print(f"Image path: {sample['image']}")
print(f"Sample sequence: {sample['sequence']}")
token_ids = tokenizer.encode(sample['sequence'])
print(f"Token IDs: {token_ids}")
print(f"Decoded: '{tokenizer.decode_to_string(token_ids)}'")


LOADING DATA
Training set: 5600 examples
Test set: 1400 examples

DATASET METADATA
Grid size: 5×5
Start position: [0, 4]
Goal position: [4, 0]
Variations per solution: 100
Seed: 42

VERIFICATION - Sample Entry
Sample ID: 0
Solution ID: 0
Variation: 0
Image path: data/grids/train/grid_0.png
Sample sequence: ['R', 'U', 'R', 'R', 'R', 'U', 'U', 'U']
Token IDs: [1, 4, 5, 4, 4, 4, 5, 5, 5, 2]
Decoded: '<s> R U R R R U U U </s>'


## 3. Create Datasets and DataLoaders

In [66]:
# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = MazeDataset(train_entries, tokenizer, transform)
test_dataset = MazeDataset(test_entries, tokenizer, transform)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
)

print(f"✓ Train loader: {len(train_loader)} batches")
print(f"✓ Test loader: {len(test_loader)} batches")

✓ Train loader: 175 batches
✓ Test loader: 44 batches


## 4. Initialize Model

In [67]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Model configuration
model = ResNetGPT2PrefixModel(
    vocab_size=len(tokenizer),
    hidden_size=128,           # Embedding dimension
    num_layers=2,              # GPT2 layers
    num_attention_heads=2,     # Attention heads
    num_prefix_tokens=8,       # Number of prefix tokens
    dropout=0.4,               # Dropout rate
    resnet_frozen=True         # Keep ResNet frozen initially
)

model = model.to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Prefix tokens: {model.num_prefix_tokens}")

Device: cuda
Parameters: 11,952,320
Prefix tokens: 8


## 5. Train Model

In [68]:
print("\n" + "=" * 60)
print("TRAINING PHASE")
print("=" * 60)
print(f"Training on {len(train_entries)} mazes for 75 epochs...")
print("=" * 60)

final_loss = train_model(model, train_loader, epochs=75, lr=5e-4, device=device)

print(f"\nTraining completed. Final loss: {final_loss:.6f}")


TRAINING PHASE
Training on 5600 mazes for 75 epochs...


Epoch 1/75: 100%|██████████| 175/175 [00:13<00:00, 13.06it/s, loss=0.5807]


Epoch 1, Avg Loss: 0.732609, LR: 5.00e-05


Epoch 2/75: 100%|██████████| 175/175 [00:13<00:00, 12.91it/s, loss=0.5420]


Epoch 2, Avg Loss: 0.557271, LR: 4.99e-05


...

Epoch 73, Avg Loss: 0.146955, LR: 1.09e-06


Epoch 74/75: 100%|██████████| 175/175 [00:14<00:00, 11.81it/s, loss=0.1246]


Epoch 74, Avg Loss: 0.140579, LR: 1.02e-06


Epoch 75/75: 100%|██████████| 175/175 [00:14<00:00, 11.73it/s, loss=0.1138]

Epoch 75, Avg Loss: 0.145909, LR: 1.00e-06

Training completed. Final loss: 0.145909


## 6. Evaluate on Training Set

In [69]:
print("\n" + "=" * 60)
print("TRAINING SET EVALUATION")
print("=" * 60)
print(f"Evaluating model performance on training set ({len(train_entries)} mazes)...")
print("=" * 60)

train_correct = test_model(model, train_loader, device, tokenizer, num_samples=len(train_entries))

print("=" * 60)
print(f"Training Accuracy: {train_correct}/{len(train_entries)} ({100*train_correct/len(train_entries):.1f}%)")
print("=" * 60)


TRAINING SET EVALUATION
Evaluating model performance on training set (5600 mazes)...
Training Accuracy: 3991/5600 (71.3%)


## 7. Evaluate on Test Set

In [70]:
print("\n" + "=" * 60)
print("TEST SET EVALUATION (UNSEEN DATA)")
print("=" * 60)
print(f"Evaluating on held-out test set ({len(test_entries)} mazes)...")
print("=" * 60)

test_correct = test_model(model, test_loader, device, tokenizer, num_samples=len(test_entries))

print("=" * 60)
print(f"Test Accuracy: {test_correct}/{len(test_entries)} ({100*test_correct/len(test_entries):.1f}%)")
print("=" * 60)


TEST SET EVALUATION (UNSEEN DATA)
Evaluating on held-out test set (1400 mazes)...
Test Accuracy: 183/1400 (13.1%)


In [71]:
# Check if model ever predicts EOS token (ID=2)
model.eval()
predictions_with_eos = 0
total_samples = 0

with torch.no_grad():
    for batch in train_loader:
        images = batch['images'].to(device)
        
        preds = model.generate(
            images, 
            max_length=12,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            bos_token_id=tokenizer.bos_token_id
        )
        
        # Check if any prediction contains EOS (token 2)
        for pred in preds:
            total_samples += 1
            if 2 in pred.cpu().tolist():
                predictions_with_eos += 1
        
        if total_samples >= 100:  # Check first 100 samples
            break

print(f"Predictions with EOS token: {predictions_with_eos}/{total_samples}")
print(f"Percentage: {100*predictions_with_eos/total_samples:.1f}%")

Predictions with EOS token: 128/128
Percentage: 100.0%


## 8. Final Results & Save Model

In [72]:
print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

train_acc_pct = 100 * train_correct / len(train_entries)
test_acc_pct = 100 * test_correct / len(test_entries)
gen_gap = train_acc_pct - test_acc_pct

print(f"Final Training Loss:    {final_loss:.6f}")
print(f"Training Accuracy:      {train_correct}/{len(train_entries)} ({train_acc_pct:.1f}%)")
print(f"Test Accuracy:          {test_correct}/{len(test_entries)} ({test_acc_pct:.1f}%)")
print(f"Generalization Gap:     {gen_gap:.1f}%")
print("=" * 60)

# Performance assessment
if final_loss < 0.1 and test_acc_pct >= 80:
    print("\n🎉 SUCCESS! Model generalizes well to unseen mazes!")
elif final_loss < 0.2 and test_acc_pct >= 60:
    print("\n✓ Good progress! Model is learning but could improve")
else:
    print("\n⚠️  Model may need more training or hyperparameter tuning")
    if gen_gap > 50:
        print("   → High generalization gap suggests overfitting")
    elif train_acc_pct < 50:
        print("   → Low training accuracy suggests underfitting or insufficient capacity")

# Save model
checkpoint_path = "models/resnet_gpt2_prefix.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': len(tokenizer),
    'hidden_size': model.hidden_size,
    'num_prefix_tokens': model.num_prefix_tokens,
    'final_loss': final_loss,
    'train_accuracy': train_acc_pct,
    'test_accuracy': test_acc_pct,
    'generalization_gap': gen_gap,
}, checkpoint_path)

print(f"\n💾 Model checkpoint saved to {checkpoint_path}")
print("=" * 60)


FINAL RESULTS SUMMARY
Final Training Loss:    0.145909
Training Accuracy:      3991/5600 (71.3%)
Test Accuracy:          183/1400 (13.1%)
Generalization Gap:     58.2%

⚠️  Model may need more training or hyperparameter tuning
   → High generalization gap suggests overfitting

💾 Model checkpoint saved to models/resnet_gpt2_prefix.pth
